In [59]:
!pip install -q google-generativeai tiktoken numpy

In [73]:
import os
import google.generativeai as genai

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Defina GEMINI_API_KEY no ambiente antes de executar o notebook.")

genai.configure(api_key=api_key)

def load_model_with_fallback(primary_model_name, fallback_model_name="models/gemini-2.5-flash"):
    try:
        print(f"Tentando carregar modelo primário: {primary_model_name}...")
        m = genai.GenerativeModel(primary_model_name)
        return m
    except Exception as e:
        print(f"Erro ao carregar {primary_model_name}: {e}")
        print(f"Ativando fallback para: {fallback_model_name}")
        return genai.GenerativeModel(fallback_model_name)

model = load_model_with_fallback("models/gemma-3-27b-it")


Tentando carregar modelo primário: gemma-4-31b-it...


In [74]:
import tiktoken

class Tokenizer:
    def __init__(self):
        try:
            self.enc = tiktoken.get_encoding("cl100k_base")
        except:
            self.enc = None

    def count(self, text):
        if self.enc:
            return len(self.enc.encode(text))
        return len(text.split())  # fallback simples

In [75]:
import numpy as np

class MiniFaiss:
    def __init__(self, dim=768):
        self.dim = dim
        self.vectors = []
        self.texts = []

    def _validate_dim(self, vector):
        if len(vector) != self.dim:
            raise ValueError(f"Dimensão inválida: esperado {self.dim}, recebido {len(vector)}")

    def add(self, vector, text):
        self._validate_dim(vector)
        self.vectors.append(vector)
        self.texts.append(text)

    def search(self, query_vector, k=3):
        self._validate_dim(query_vector)
        sims = []
        for i, v in enumerate(self.vectors):
            sim = np.dot(query_vector, v) / (
                np.linalg.norm(query_vector) * np.linalg.norm(v) + 1e-9
            )
            sims.append((sim, self.texts[i]))

        sims.sort(reverse=True)
        return sims[:k]


In [76]:
class TurboQuant:
    def __init__(self, bits=8):
        self.bits = bits
        self.scale = None

    def fit(self, vectors):
        max_val = np.max(np.abs(vectors))
        self.scale = max_val / (2**(self.bits - 1) - 1)

    def quantize(self, vector):
        return np.round(vector / self.scale).astype(np.int8)

    def dequantize(self, q_vector):
        return q_vector.astype(np.float32) * self.scale


def quantization_error(original, reconstructed):
    return np.mean((original - reconstructed) ** 2)


def compression_ratio(original_bits=32, quant_bits=8):
    return original_bits / quant_bits

In [77]:
import time

class Trace:
    def __init__(self):
        self.steps = []

    def log(self, step, data):
        self.steps.append({
            "step": step,
            "time": time.time(),
            "data": data
        })

    def show(self):
        for s in self.steps:
            print(s)

In [78]:
class GeminiLLM:
    def __init__(self, model):
        self.model = model

    def generate(self, prompt):
        response = self.model.generate_content(prompt)
        return response.text

In [79]:
# Optimized embedding function using Gemini 001
def embed_text(text):
    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=text,
        output_dimensionality=768  # 🔥 economia + velocidade
    )
    return np.array(result["embedding"])

In [80]:
class Harness:
    def __init__(self, llm, retriever, tokenizer):
        self.llm = llm
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.trace = Trace()

    def embed(self, text):
        return embed_text(text)

    def run(self, query):
        query_vec = self.embed(query)

        docs = self.retriever.search(query_vec)
        self.trace.log("retrieval", docs)

        context = "\n".join([d[1] for d in docs])

        prompt = f"""
        Contexto:
        {context}

        Pergunta:
        {query}
        """

        response = self.llm.generate(prompt)
        self.trace.log("llm_response", response)

        tokens = self.tokenizer.count(prompt + response)

        return {
            "response": response,
            "tokens": tokens,
            "docs": docs
        }


In [81]:
docs = [
    "TurboQuant reduz custo de embeddings",
    "MiniFaiss permite busca vetorial leve",
    "Gemini 3 tem raciocínio avançado",
    "LLMs podem ser avaliados com harness"
]

In [83]:
import time

retriever = MiniFaiss()

# Populando com embeddings REAIS e quantizados
for d in docs:
    success = False
    retries = 3
    while retries > 0 and not success:
        try:
            original_vec = embed_text(d)

            if 'tq' not in globals():
                tq = TurboQuant()

            if tq.scale is None:
                tq.fit(np.array([original_vec]))

            q_vec = tq.quantize(original_vec)
            dq_vec = tq.dequantize(q_vec)
            retriever.add(dq_vec, d)

            print(f"Sucesso: '{d}'")
            success = True
            time.sleep(2) # Pausa maior para evitar 429

        except Exception as e:
            if "429" in str(e):
                print(f"⚠️ Limite atingido em '{d}'. Aguardando 10s... (Tentativas restantes: {retries})")
                time.sleep(10)
                retries -= 1
            else:
                print(f"Erro inesperado em '{d}': {e}")
                break

print("\nRetriever atualizado e pronto.")

Sucesso: 'TurboQuant reduz custo de embeddings'
Sucesso: 'MiniFaiss permite busca vetorial leve'
Sucesso: 'Gemini 3 tem raciocínio avançado'
Sucesso: 'LLMs podem ser avaliados com harness'

Retriever atualizado e pronto.


In [84]:
vectors = np.random.rand(10, 128)

tq = TurboQuant()
tq.fit(vectors)

q = tq.quantize(vectors[0])
dq = tq.dequantize(q)

print("Erro:", quantization_error(vectors[0], dq))
print("Compressão:", compression_ratio())

Erro: 4.629075532420056e-06
Compressão: 4.0


In [85]:
tokenizer = Tokenizer()
llm = GeminiLLM(model)

harness = Harness(llm, retriever, tokenizer)

result = harness.run("O que é TurboQuant?")

print("Resposta:\n", result["response"])
print("\nTokens:", result["tokens"])

print("\nTRACE:")
harness.trace.show()

ValueError: shapes (128,) and (768,) not aligned: 128 (dim 0) != 768 (dim 0)

In [72]:
# Listar modelos disponíveis para encontrar o nome correto
print("Modelos que suportam geração de conteúdo:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"ID: {m.name} | Display Name: {m.display_name}")

Modelos que suportam geração de conteúdo:
ID: models/gemini-2.5-flash | Display Name: Gemini 2.5 Flash
ID: models/gemini-2.5-pro | Display Name: Gemini 2.5 Pro
ID: models/gemini-2.0-flash | Display Name: Gemini 2.0 Flash
ID: models/gemini-2.0-flash-001 | Display Name: Gemini 2.0 Flash 001
ID: models/gemini-2.0-flash-lite-001 | Display Name: Gemini 2.0 Flash-Lite 001
ID: models/gemini-2.0-flash-lite | Display Name: Gemini 2.0 Flash-Lite
ID: models/gemini-2.5-flash-preview-tts | Display Name: Gemini 2.5 Flash Preview TTS
ID: models/gemini-2.5-pro-preview-tts | Display Name: Gemini 2.5 Pro Preview TTS
ID: models/gemma-3-1b-it | Display Name: Gemma 3 1B
ID: models/gemma-3-4b-it | Display Name: Gemma 3 4B
ID: models/gemma-3-12b-it | Display Name: Gemma 3 12B
ID: models/gemma-3-27b-it | Display Name: Gemma 3 27B
ID: models/gemma-3n-e4b-it | Display Name: Gemma 3n E4B
ID: models/gemma-3n-e2b-it | Display Name: Gemma 3n E2B
ID: models/gemma-4-26b-a4b-it | Display Name: Gemma 4 26B A4B IT
ID: m

In [ ]:
def recall_at_k(retrieved, expected):
    return any(expected in r[1] for r in retrieved)

In [ ]:
# Testando o Recall com o novo setup
query = "Como reduzir custo de embeddings?"
expected_substring = "TurboQuant"

# Executar busca
result_bench = harness.run(query)
retrieved_docs = result_bench["docs"]

# Avaliar
success = recall_at_k(retrieved_docs, expected_substring)

print(f"Busca por: '{query}'")
print(f"Documentos recuperados: {[d[1] for d in retrieved_docs]}")
print(f"Recall@K Sucesso: {success}")

# Comparar erro de quantização médio do banco
all_vecs = np.array([embed_text(d) for d in docs])
q_all = tq.quantize(all_vecs)
dq_all = tq.dequantize(q_all)
erro_medio = np.mean([quantization_error(all_vecs[i], dq_all[i]) for i in range(len(all_vecs))])
print(f"Erro de quantização médio no dataset: {erro_medio:.6f}")